# Extracting Embeddings

Vision model for middle frames and generated images: DINOv2 (large and base)

Audio model: BEATs (iter3 and iter3+)

Language model: Qwen3-1.7B

In [ ]:
# libraries
import os
import pandas as pd
import numpy as np
import torch
import torchaudio
from tqdm import tqdm
import shutil
from google.colab import files
from PIL import Image
from transformers import AutoImageProcessor, AutoProcessor, AutoModel, AutoTokenizer, AutoModelForCausalLM

In [ ]:
!git clone https://github.com/microsoft/unilm.git

import sys
sys.path.append("/content/unilm/beats")

from BEATs import BEATs, BEATsConfig

# manually add beats iter3 and iter3+ checkpoints to beats folder

Cloning into 'unilm'...
remote: Enumerating objects: 11207, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 11207 (delta 3), reused 2 (delta 2), pack-reused 11198 (from 2)
Receiving objects: 100% (11207/11207), 75.40 MiB | 26.48 MiB/s, done.
Resolving deltas: 100% (5312/5312), done.


In [ ]:
print(torch.__version__)

2.10.0+cu128


In [ ]:
# make folders
os.makedirs('images/middle_frames', exist_ok=True)
os.makedirs('audio', exist_ok=True)
os.makedirs('beats', exist_ok=True)

# manually upload csv
# manually upload vision and audio files
# manually add beats iter3 and iter3plus checkpoints to beats folder

In [ ]:
# unzip generated images
shutil.unpack_archive('generated_images.zip', 'unzipped_gen_images', 'zip')
shutil.move('/content/unzipped_gen_images/generated_images', 'images')

# make sure in correct folder (images/generated_images)

'images/generated_images'

In [ ]:
df = pd.read_csv("/content/final_audiocaps.csv")
print(df.shape)
df.head()

(620, 7)


,Unnamed: 0,audiocap_id,youtube_id,start_time,caption,audio,video
0,76,105889,VQnmlf2OsUg,50.0,helicopter blades spinning,1,1
1,240,102999,KnXNy5Q6YS4,230.0,muffled speech and insect buzzing,1,1
2,524,103149,D4s5aHrsBgs,80.0,music is playing as a person whistles,1,1
3,585,105559,c3nlaAkv9bA,230.0,male speech and a goat bleating,1,1
4,1166,103522,eYbFtxZmKL4,110.0,clip-clops from a horse in the distant with so...,1,1


## Prepare Functions

In [ ]:
# vision model functions
def load_dinov2(model_size='large', device=None):
  # model mapping
  model_map = {
      'small': 'facebook/dinov2-small',
      'base': 'facebook/dinov2-base',
      'large': 'facebook/dinov2-large',
      'giant': 'facebook/dino2-giant'
  }

  if model_size not in model_map:
      raise ValueError(f'model_size must be one of {list(model_map.keys())}')

  model_name = model_map[model_size]

  if device is None:
      device = 'cuda' if torch.cuda.is_available() else 'cpu'

  # load processor and model
  processor = AutoImageProcessor.from_pretrained(model_name)
  model = AutoModel.from_pretrained(model_name).to(device)
  model.eval()

  return processor, model, device


def extract_dinov2_embeddings(df, image_root, save_root, processor, model, device, pooling='mean', gen_image=True, overwrite=False):
  """
  Extract DINOv2 embeddings from images and save as .npy files
  """
  # create save root
  os.makedirs(save_root, exist_ok=True)

  # loop through images
  for _, row in tqdm(df.iterrows(), total=len(df)):
      image_id = str(row['audiocap_id'])

      # correct image path depending on file structures
      if gen_image == True:
        img_path = os.path.join(image_root, image_id, '0.png')
      else:
        img_path = os.path.join(image_root, f'{image_id}.png')

      save_path = os.path.join(save_root, f'{image_id}.npy')

      # skip missing files
      if not os.path.exists(img_path):
        print(f'Missing image: {img_path}')
        continue

      # skip existing embeddings
      if os.path.exists(save_path) and not overwrite:
        continue

      # load image
      image = Image.open(img_path).convert('RGB')

      # preprocess image
      inputs = processor(images=image, return_tensors='pt')
      inputs = {k: v.to(device) for k, v in inputs.items()}

      # extract embeddings
      with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

        if pooling == 'cls':
          # CLS token from each layer
          layer_embeddings = torch.stack([
              h[:, 0, :] for h in outputs.hidden_states
          ])

        elif pooling == 'mean':
          # mean over all patch tokens
          layer_embeddings = torch.stack([
              h.mean(dim=1) for h in outputs.hidden_states
          ])

        else:
          raise ValueError("pooling must be 'cls' or 'mean'")

        embedding = layer_embeddings.squeeze().cpu().numpy()

      # save embedding
      np.save(save_path, embedding)

  print('Embedding extraction complete!')


In [ ]:
# audio model functions
def load_beats_model(checkpoint_path):
  checkpoint = torch.load(checkpoint_path, map_location=device)

  cfg = BEATsConfig(checkpoint["cfg"])
  model = BEATs(cfg)
  model.load_state_dict(checkpoint["model"])

  model = model.to(device)
  model.eval()

  return model

def extract_beats_embeddings(model, df, audio_root, save_root, device, target_sr=16000, segment_sec=10):
  os.makedirs(save_root, exist_ok=True)

  hidden_states = []

  def hook_fn(module, input, output):
    if isinstance(output, tuple):
      hidden_states.append(output[0].detach().cpu())
    else:
      hidden_states.append(output.detach().cpu())

  # register hooks
  handles = []
  for layer in model.encoder.layers:
    handles.append(layer.register_forward_hook(hook_fn))

  segment_len = segment_sec * target_sr

  for _, row in tqdm(df.iterrows(), total=len(df)):
      audio_id = str(row['audiocap_id'])

      audio_path = os.path.join(audio_root, f'{audio_id}.wav')
      save_path = os.path.join(save_root, f'{audio_id}.npy')

      if not os.path.exists(audio_path):
        print(f'Missing audio for {audio_id}')
        continue

      if os.path.exists(save_path):
        continue

      waveform, sr = torchaudio.load(audio_path)
      waveform = waveform.mean(dim=0)

      if sr != target_sr:
        waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)

      mid = waveform.shape[0] // 2
      waveform = waveform[max(0, mid - segment_len // 2): mid + segment_len // 2]
      waveform = waveform.to(device)

      hidden_states.clear()

      with torch.no_grad():
        _ = model.extract_features(waveform.unsqueeze(0))

      if len(hidden_states) == 0:
        print(f'No layers capture for {audio_id}')

      layer_embeddings = torch.stack([
          h.squeeze(1).mean(dim=0) for h in hidden_states
      ])

      np.save(save_path, layer_embeddings.numpy().astype(np.float32))

  # clean up hooks
  for h in handles:
    h.remove()

In [ ]:
# language model functions
def extract_qwen3_embeddings(df, text_col, save_root, tokenizer, model, device, pooling='mean', overwrite=False):
  """
  Extract Qwen 3 per-layer embeddings and save as .npy files
  """
  os.makedirs(save_root, exist_ok=True)

  for _, row in tqdm(df.iterrows(), total=len(df)):
      text_id = str(row['audiocap_id'])
      text = row[text_col]

      save_path = os.path.join(save_root, f'{text_id}.npy')

      if os.path.exists(save_path) and not overwrite:
          continue

      # tokenize
      inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
      inputs = {k: v.to(device) for k, v in inputs.items()}

      with torch.no_grad():
          outputs = model(**inputs, output_hidden_states=True, return_dict=True)

          hidden_states = outputs.hidden_states
          mask = inputs['attention_mask']

          if pooling == 'mean':
            layer_embeddings = torch.stack([
                (h * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True) for h in hidden_states
            ])

          elif pooling == 'cls':
            layer_embeddings = torch.stack([
                  h[:, 0, :] for h in hidden_states
              ])

          elif pooling == "last":
            layer_embeddings = torch.stack([
                  h[:, -1, :] for h in hidden_states
              ])

          else:
            raise ValueError("pooling must be 'mean', 'cls', or 'last'")

          embedding = layer_embeddings.squeeze(1).cpu().float().numpy()

      np.save(save_path, embedding)
  print('Qwen3 embedding extraction complete!')

In [ ]:
def get_layer_embeddings(input_folder, save_root, num_layers, emb_shape):
  """
  Get layer-wise embeddings and save as .npy files

  parameters:
  input_folder (string): file path to embeddings folder
  save_root (string): file path to output folder
  num_layers (integer): number of hidden layers in model embeddings
  emb_shape (tuple): correct embedding shape
  """

  os.makedirs(save_root, exist_ok=True)

  # sorted list of embedding files in folder
  files = sorted([f for f in os.listdir(input_folder) if f.endswith(".npy") and f!= "ids.npy"])
  ids = [f.replace(".npy", "") for f in files]
  np.save(os.path.join(save_root, "ids.npy"), np.array(ids))

  # process one layer at a time
  for l in range(num_layers):
    print(f"\nProcessing layer {l}")

    layer_vectors = []

    for f in tqdm(files):
        emb = np.load(os.path.join(input_folder, f))

        if emb.shape != emb_shape:
            raise ValueError(f"{f} has wrong shape {emb.shape}")

        layer_vectors.append(emb[l])

    layer_matrix = np.stack(layer_vectors).astype(np.float32)

    np.save(os.path.join(save_root, f"layer_{l}.npy"), layer_matrix)

  print("Layer embeddings saved!")

# Vision Embeddings (DINOv2)

## DINOv2 Large

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# load dinov2 large
processor_large, model_large, device = load_dinov2(model_size='large', device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

In [ ]:
# middle frames
extract_dinov2_embeddings(
    df,
    image_root="/content/images/middle_frames",
    save_root="/content/dinov2/large/frame_embeddings",
    processor=processor_large,
    model=model_large,
    device=device,
    pooling='mean',
    gen_image=False
)

# layer embeddings
get_layer_embeddings(
    input_folder="/content/dinov2/large/frame_embeddings",
    save_root="/content/dinov2/large/frame_layers",
    num_layers=25,
    emb_shape=(25,1024)
)

100%|██████████| 620/620 [00:48<00:00, 12.88it/s]


Embedding extraction complete!

Processing layer 0


100%|██████████| 620/620 [00:00<00:00, 3054.79it/s]



Processing layer 1


100%|██████████| 620/620 [00:00<00:00, 4092.93it/s]


...


Processing layer 23


100%|██████████| 620/620 [00:00<00:00, 7051.70it/s]



Processing layer 24


100%|██████████| 620/620 [00:00<00:00, 4532.22it/s]

Layer embeddings saved!


In [ ]:
# generated images
extract_dinov2_embeddings(
    df,
    image_root="/content/images/generated_images",
    save_root="/content/dinov2/large/gen_embeddings",
    processor=processor_large,
    model=model_large,
    device=device,
    pooling='mean',
    gen_image=True
)

# layer embeddings
get_layer_embeddings(
    input_folder="/content/dinov2/large/gen_embeddings",
    save_root="/content/dinov2/large/gen_layers",
    num_layers=25,
    emb_shape=(25,1024)
)

100%|██████████| 620/620 [01:00<00:00, 10.17it/s]


Embedding extraction complete!

Processing layer 0


100%|██████████| 620/620 [00:00<00:00, 6564.15it/s]



Processing layer 1


100%|██████████| 620/620 [00:00<00:00, 7994.33it/s]


...


Processing layer 23


100%|██████████| 620/620 [00:00<00:00, 6233.56it/s]



Processing layer 24


100%|██████████| 620/620 [00:00<00:00, 5580.38it/s]

Layer embeddings saved!


In [ ]:
# zip embeddings
shutil.make_archive('frame_large_embeddings', 'zip', '/content/dinov2/large/frame_embeddings')
shutil.make_archive('frame_large_layers', 'zip', '/content/dinov2/large/frame_layers')

shutil.make_archive('gen_large_embeddings', 'zip', '/content/dinov2/large/gen_embeddings')
shutil.make_archive('gen_large_layers', 'zip', '/content/dinov2/large/gen_layers')

'/content/gen_large_layers.zip'

## DINOv2 Base

In [ ]:
# load dinov2 base
processor_base, model_base, device = load_dinov2(model_size='base', device=device)

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [ ]:
# middle frames
extract_dinov2_embeddings(
    df,
    image_root="/content/images/middle_frames",
    save_root="/content/dinov2/base/frame_embeddings",
    processor=processor_base,
    model=model_base,
    device=device,
    pooling='mean',
    gen_image=False
)

# layer embeddings
get_layer_embeddings(
    input_folder="/content/dinov2/base/frame_embeddings",
    save_root="/content/dinov2/base/frame_layers",
    num_layers=13,
    emb_shape=(13,768)
)

In [ ]:
# generated images
extract_dinov2_embeddings(
    df,
    image_root="/content/images/generated_images",
    save_root="/content/dinov2/base/gen_embeddings",
    processor=processor_base,
    model=model_base,
    device=device,
    pooling='mean',
    gen_image=True
)

# layer embeddings
get_layer_embeddings(
    input_folder="/content/dinov2/base/gen_embeddings",
    save_root="/content/dinov2/base/gen_layers",
    num_layers=13,
    emb_shape=(13,768)
)

100%|██████████| 620/620 [00:47<00:00, 13.08it/s]


Embedding extraction complete!

Processing layer 0


100%|██████████| 620/620 [00:00<00:00, 5576.73it/s]



Processing layer 1


100%|██████████| 620/620 [00:00<00:00, 5283.23it/s]


...


Processing layer 11


100%|██████████| 620/620 [00:00<00:00, 10296.52it/s]



Processing layer 12


100%|██████████| 620/620 [00:00<00:00, 10156.29it/s]

Layer embeddings saved!


In [ ]:
# zip embeddings
shutil.make_archive('frame_base_embeddings', 'zip', '/content/dinov2/base/frame_embeddings')
shutil.make_archive('frame_base_layers', 'zip', '/content/dinov2/base/frame_layers')

shutil.make_archive('gen_base_embeddings', 'zip', '/content/dinov2/base/gen_embeddings')
shutil.make_archive('gen_base_layers', 'zip', '/content/dinov2/base/gen_layers')

'/content/gen_base_layers.zip'

# Audio Embeddings (BEATs)

Load BEATs models (iter3 and iter3+).

Use forward hooks to extract layer embeddings from BEATs models.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# BEATs iter3+
beats3_plus = load_beats_model('beats/BEATs_iter3_plus_AS2M.pt')
extract_beats_embeddings(
    model=beats3_plus,
    df=df,
    audio_root="audio",
    save_root="beats/iter3_plus/audio_embeddings",
    device=device
)

# BEATS iter3
beats3 = load_beats_model('/content/beats/BEATs_iter3.pt')
extract_beats_embeddings(
    model=beats3,
    df=df,
    audio_root="audio",
    save_root="beats/iter3/audio_embeddings",
    device=device
)

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
100%|██████████| 620/620 [00:42<00:00, 14.57it/s]


In [ ]:
# testing
test_emb_3 = np.load("/content/beats/iter3/audio_embeddings/102999.npy")
test_emb_plus = np.load("/content/beats/iter3_plus/audio_embeddings/102999.npy")
print(test_emb_3.shape, test_emb_plus.shape)

(12, 768) (12, 768)


In [ ]:
# BEATs iter3+
get_layer_embeddings(
    input_folder="/content/beats/iter3_plus/audio_embeddings",
    save_root="/content/beats/iter3_plus/audio_layers",
    num_layers=12,
    emb_shape=(12,768)
)

# BEATs iter3
get_layer_embeddings(
    input_folder="/content/beats/iter3/audio_embeddings",
    save_root="/content/beats/iter3/audio_layers",
    num_layers=12,
    emb_shape=(12,768)
)


Processing layer 0


100%|██████████| 620/620 [00:00<00:00, 9380.29it/s]



Processing layer 1


100%|██████████| 620/620 [00:00<00:00, 10072.58it/s]



Processing layer 2


...


Processing layer 10


100%|██████████| 620/620 [00:00<00:00, 10185.37it/s]



Processing layer 11


100%|██████████| 620/620 [00:00<00:00, 10427.19it/s]

Layer embeddings saved!


In [ ]:
# zip embeddings
shutil.make_archive('iter3_embeddings', 'zip', '/content/beats/iter3/audio_embeddings')
shutil.make_archive('iter3_plus_embeddings', 'zip', '/content/beats/iter3_plus/audio_embeddings')

shutil.make_archive('iter3_layers', 'zip', '/content/beats/iter3/audio_layers')
shutil.make_archive('iter3_plus_layers', 'zip', '/content/beats/iter3_plus/audio_layers')

'/content/iter3_plus_layers.zip'

# Language Embeddings

In [ ]:
def load_qwen3_model(model_name='Qwen/Qwen3-8B'):
  """
  Load Qwen3 model and tokenizer with hidden states enabled.
  """
  tokenizer = AutoTokenizer.from_pretrained(model_name)

  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      torch_dtype=torch.float32,
      device_map='auto'
  )

  model.eval()

  return tokenizer, model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# load model

model_name='Qwen/Qwen3-1.7B'

qwen3_tokenizer = AutoTokenizer.from_pretrained(model_name)

qwen3_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map='auto',
)

# qwen3_model = qwen3_model.to(device)
qwen3_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

In [ ]:
# extract embeddings
extract_qwen3_embeddings(
    df=df,
    text_col='caption',
    save_root='qwen3-1.7b/text_embeddings',
    tokenizer=qwen3_tokenizer,
    model=qwen3_model,
    device=device
)

100%|██████████| 620/620 [00:36<00:00, 16.94it/s]

Qwen3 embedding extraction complete!


In [ ]:
# check embedding shape
np.load('qwen3-1.7b/text_embeddings/100110.npy').shape

(29, 2048)

In [ ]:
# qwen3 layer embeddings
get_layer_embeddings(
    input_folder="qwen3-1.7b/text_embeddings",
    save_root="qwen3-1.7b/text_layers",
    num_layers=29,
    emb_shape=(29,2048)
)


Processing layer 0


100%|██████████| 620/620 [00:00<00:00, 3466.86it/s]



Processing layer 1


100%|██████████| 620/620 [00:00<00:00, 5729.85it/s]



Processing layer 2


...


Processing layer 27


100%|██████████| 620/620 [00:00<00:00, 5542.23it/s]



Processing layer 28


100%|██████████| 620/620 [00:00<00:00, 5746.21it/s]

Layer embeddings saved!


In [ ]:
# zip embeddings
shutil.make_archive('qwen3_1.7b_embeddings', 'zip', 'qwen3-1.7b/text_embeddings')
shutil.make_archive('qwen3_1.7b_layers', 'zip', 'qwen3-1.7b/text_layers')

'/content/qwen3_0.6b_layers.zip'

# Checking order of layer embeddings

In [ ]:
frame_base_ids = np.load("/content/dinov2/base/frame_layers/ids.npy")
frame_large_ids = np.load("/content/dinov2/large/frame_layers/ids.npy")
gen_base_ids = np.load("/content/dinov2/base/gen_layers/ids.npy")
gen_large_ids = np.load("/content/dinov2/large/gen_layers/ids.npy")
audio3_ids = np.load("/content/beats/iter3/audio_layers/ids.npy")
audio3_plus_ids = np.load("/content/beats/iter3_plus/audio_layers/ids.npy")
qwen3_ids = np.load("/content/qwen3-1.7b/text_layers/ids.npy")

embedding_ids = [frame_base_ids, frame_large_ids, gen_base_ids, gen_large_ids, audio3_ids, audio3_plus_ids, qwen3_ids]
embedding_ids_names = ["frame_base_ids", "frame_large_ids", "gen_base_ids", "gen_large_ids", "audio3_ids", "audio3_plus_ids", "qwen3_ids"]

pair_names = list(itertools.combinations(embedding_ids_names, 2))

for name, pair in zip(pair_names, itertools.combinations(embedding_ids, 2)):
    if np.array_equal(pair[0], pair[1]):
        print(f'Same id order for {name}')
    else:
        print(f'*** Error: different order for {name} ***')